![bse_logo_textminingcourse](https://bse.eu/sites/default/files/bse_logo_small.png)

# Part 1: Scraping

Design and implement a mini research project in which you research the effect of a big
annual event in Barcelona on rental prices on booking by scraping data for at least two
separate weeks (important note that search results go across different pages) for Barcelona
and at least one more city.

1. Identify a (future) event that makes a lot of people come to Barcelona. Think about
music festivals, local festivities etc.
2. Think of the time periods to scrape and what second city to scrape. The second city
will be your control group. Explain your choices in written.
3. Design a careful scraping pipeline that follows the advises seen in class and TAs. The basic points to bear in mind are:
    - Organize the data you need, format and structure to store it beforehand. Try to foresee how you will need to read in the data to answer your questions. If you want, you can include some few lines explaining your pipeline strategy at the beginning.
    - Codes should be as automated as possible. That is, you don’t want to rely on human intervention to get your data.
    - Use only the packages we have seen in the course. Although firefox is recommended, you can also use chrome as your scraping browser.
    - Document your codes and make them robust and efficient.

**Chosen event**: Sant Jordi (23rd April).

Alternatives:
- La Mercè (24th September).
- Primavera Sound.
- Mobile World Congress (February).

**Second city to scrape**: Madrid.

## 0. Packages

In [22]:
from selenium.webdriver.common.by import By
from selenium.common.exceptions import (
    NoSuchElementException,
    ElementClickInterceptedException,
    StaleElementReferenceException,
)
from selenium import webdriver
import os
import time
from selenium import webdriver
from selenium.webdriver.firefox.service import Service
from selenium.webdriver.firefox.options import Options

# We import a module that allows to input keys
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains

# Go get geckodriver from : https://github.com/mozilla/geckodriver/releases

## 1. Data to be scraped

The following attributes may be of interest for this homework:
- Name of the hotel/apartment.
- Price.
- Characteristics of the accomodation (description). E.g., "Habitación Doble con baño privado - 1 o 2 camas; Camas: 1 doble o 2 individuales".

Secondary attributes:
- Number of stars (hotels) or apartment quality (yellow squares with a white dot inside).
- Neighborhood.
- Number of comments.

## 2. Utilities

In [23]:
# Function for defining the Firefox preferences: we initialize a blank browser
# with a new profile
def ffx_preferences(dfolder, download=False):
    """
    Sets the preferences of the firefox browser: download path.
    """
    profile = webdriver.FirefoxProfile()
    # set download folder:
    profile.set_preference(
        "browser.download.dir", dfolder
    )  # you can predefine where you wanna store things in case its needed
    profile.set_preference(
        "browser.download.folderList", 2
    )  # 0 means to download to the desktop, 1 means to download to the default "Downloads" directory, 2 means to use the directory
    profile.set_preference(
        "browser.download.manager.showWhenStarting", False
    )  # I dont wanna see a pop up for each download so i swicth it off
    profile.set_preference(
        "browser.helperApps.neverAsk.saveToDisk",
        "application/msword,application/rtf, application/csv,text/csv,image/png ,image/jpeg, application/pdf, text/html,text/plain,application/octet-stream",
    )

    # profile.install_addon('/Users/luisignaciomenendezgarcia/Dropbox/CLASSES/class_bse_text_mining/class_scraping_bse_2025/booking/booking/ublock_origin-1.55.0.xpi')
    # profile.add_extension('/Users/luisignaciomenendezgarcia/Dropbox/CLASSES/class_bse_text_mining/class_scraping_bse/booking/booking/ublock_origin-1.55.0.xpi')

    # this allows to download pdfs automatically
    if download:
        profile.set_preference(
            "browser.helperApps.neverAsk.saveToDisk",
            "application/pdf,application/x-pdf",
        )
        profile.set_preference(
            "pdfjs.disabled", True
        )  # dont want the pdf viewer to open

    options = Options()
    options.profile = profile
    # We indicate the location of the Firefox executable
    # options.binary_location = r"/usr/bin/firefox"
    return options


def start_up(link, dfolder, geko_path, download=True):
    os.makedirs(dfolder, exist_ok=True)

    # Set TMPDIR environment variable for sandboxed Firefox (Snap package, for
    # Ubuntu users)
    os.environ["TMPDIR"] = os.path.expanduser("~/tmp")

    # Get Firefox options, including download preferences if applicable
    options = ffx_preferences(dfolder, download)

    service = Service(geko_path)
    browser = webdriver.Firefox(service=service, options=options)

    # Enter the website address here
    browser.get(link)
    time.sleep(5)  # Adjust sleep time as needed
    return browser


def check_and_click(browser, xpath, type):
    """
    Function that checks whether the object is clickable and, if so, clicks on
    it. If not, waits one second and tries again.
    """
    ck = False
    ss = 0
    while ck == False:
        ck = check_obscures(browser, xpath, type)
        time.sleep(1)
        ss += 1
        if ss == 15:
            # warn_sound()
            # return NoSuchElementException
            ck = True
            # browser.quit()


def check_obscures(browser, xpath, type):
    """
    Function that checks whether the object is being "obscured" by any element so
    that it is not clickable. Important: if True, the object is going to be clicked!
    """
    try:
        if type == "xpath":
            browser.find_element("xpath", xpath).click()
        elif type == "id":
            browser.find_element("id", xpath).click()
        elif type == "css":
            browser.find_element("css selector", xpath).click()
        elif type == "class":
            browser.find_element("class name", xpath).click()
        elif type == "link":
            browser.find_element("link text", xpath).click()
    except (
        ElementClickInterceptedException,
        NoSuchElementException,
        StaleElementReferenceException,
    ) as e:
        print(e)
        return False
    return True

def scroll_and_click(browser, by_type, path):
    """
    Scrolls until the given element is visible and clicks on it.

    Args:
        browser: Selenium WebDriver instance created using the `start_up` function.
        by_type: The type of locator to use (e.g., 'xpath', 'css selector', etc.).
        path: The path to locate the element based on the specified locator type.
    """
    try:
        # Find the element using the specified locator type
        element = browser.find_element(by=by_type, value=path)
        
        # Scroll until the element is visible
        browser.execute_script(
            "arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", element
        )
        
        # Click on the element
        element.click()
    except Exception as e:
        print(f"Error while trying to scroll and click: {e}")

def scroll_and_click_dates(browser, by_type, path, start_date, end_date):
    """
    Scrolls through a calendar, finds specified dates, and clicks on them.

    Args:
        browser: Selenium WebDriver instance.
        by_type: Locator type (e.g., 'xpath', 'css selector', etc.).
        path: Locator path to identify all date elements.
        start_date: Start date as a string (e.g., "2025-01-24").
        end_date: End date as a string (e.g., "2025-01-31").
    """
    try:
        # Find all dates matching the given path
        dates = browser.find_elements(by=by_type, value=path)
        
        # Iterate over the dates and click the start and end dates
        for date in dates:
            current_date = date.get_attribute("data-date")
            if current_date == start_date or current_date == end_date:
                # Scroll to the specific date
                browser.execute_script(
                    "arguments[0].scrollIntoView({behavior: 'smooth', block: 'center'});", date
                )
                # Click on the date
                date.click()
                # Stop if the end date is clicked
                if current_date == end_date:
                    break
    except Exception as e:
        print(f"Error while scrolling and clicking on dates: {e}")

def scroll_until_button_visible(browser, button_xpath, scroll_pause=1, max_attempts=50):
    """
    Scrolls down the page until the "Load more results" button becomes visible.

    This function repeatedly scrolls down the page by the height of the visible window 
    using `window.scrollBy(0, window.innerHeight)`. After each scroll, it checks whether 
    the specified button is visible. If the button is found and displayed, the function 
    exits successfully. A pause (`scroll_pause`) is added between scrolls to allow time 
    for additional content to load. To avoid infinite loops, a maximum number of scroll 
    attempts (`max_attempts`) is defined.

    Args:
        browser: Selenium WebDriver instance.
        button_xpath: XPath of the "Load more results" button to locate.
        scroll_pause: Time (in seconds) to pause between scroll actions. Default is 1 second.
        max_attempts: Maximum number of scroll attempts before stopping. Default is 50.

    Returns:
        bool: True if the button is found and visible, False otherwise.

    Example Usage:
        path = '//div[@class="c82435a4b8 f581fde0b8"]//button[@class="a83ed08757 c21c56c305 bf0537ecb5 f671049264 af7297d90d c0e0affd09"]'

        if scroll_until_button_visible(browser, path):
            scroll_and_click(browser=browser, by_type='xpath', path=path)
        else:
            print("Load more results button not found.")
    """
    attempts = 0
    button_found = False

    while attempts < max_attempts:
        # Scroll down to load more results
        browser.execute_script("window.scrollBy(0, window.innerHeight);")
        time.sleep(scroll_pause)  # Pause to allow loading of content

        try:
            # After scrolling, try to find the button
            button = browser.find_element(by="xpath", value=button_xpath)
            if button.is_displayed():
                button_found = True
                break  # Button found and clicked
        except:
            pass  # Ignore errors if the button is not found yet

        attempts += 1

    if button_found:
        return True
    else:
        print("Reached maximum attempts or button not found.")
        return False

def load_all_results(browser, button_xpath, scroll_pause=1, max_attempts=50):
    # Scroll down and click in the Load more results" button

    # Initialize variables
    click_count = 0  # Counter for the number of button clicks

    # Loop until button is no longer visible
    while True:
        if scroll_until_button_visible(browser, button_xpath):
            scroll_and_click(browser=browser, by_type='xpath', path=button_xpath)
            click_count += 1
            
        else:
            break  # Exit the loop if no button is found

    # Print the final click count
    print(f"Total 'Load more results' button clicks: {click_count}.")

Need to adjust the following function, which does not scroll more than once:

## 3. Search hierarchy

When searching for elements in HTML, it's common to use either XPath or CSS selectors. The choice of selector depends on the specific requirements and context of the task. Below is a basic guide to understanding the structure of these selectors.

1. **ID**: 
   - **Description**: Unique identifier for an element. Not always available.
   - **Format**: `id="unique_id"`
2. **Class**:
   - **Description**: Class name(s) associated with an element.
   - **Format**: `class="class_name"`
3. **Tag**:
   - **Description**: HTML tag (like `div`, `label`, etc.).
   - **Format**: `<tag>`
4. **Attribute**:
   - **Description**: An attribute and its value within an element.
   - **Format**: `attribute="attribute_value"`

### XPath

- **Basic Structure**:
  
  ```xpath
  //tag[@attribute="attribute_value"][position]

The double bar `//` selects all elements that match that path. As in regex, you can use * to match any element

### CSS

You also have CSS selectors that will match elements based on attributes. These are easier but less robust than xpath. Sometimes they will allow you to differentitate HTML elements that would otherwise be equal

``` <span class ="green"></span>
<span class ="red"></span>

With their class, you can get say all the links that are green but not red. 

## 4. Organization of the data

Each row of the data frame can be one data extraction with the following information:
- Name of the accommodation.
- Price.
- Date of beginning of rental.
- Date of end of rental.
- Description of the accommodation.
- Number of stars.
- Neighborhood.
- Description of the accommodation.
- Number of comments.

## 5. Scraping

In [58]:
# We set as the download directory the folder 'files',
# defined through a relative path
dfolder = "./files"
# Set the geckodriver path
geko_path = "/usr/local/bin/geckodriver"
# Link to booking
link = "https://www.booking.com/index.es.html"

browser = start_up(dfolder=dfolder, link=link, geko_path=geko_path)

In [59]:
# We have to reject the cookies to simplify the process of putting buttons into
# view (which is useful when we want to click a button that may be obscured 
# otherwise)
path = '//*[@id="onetrust-reject-all-handler"]'
browser.find_element(by="xpath", value= path).click()

In [60]:
# We scroll and click on the "Where are you going?" search button
scroll_and_click(browser = browser, by_type = 'xpath', path = '//*[@id=":rh:"]')
# Input the destination for which you want to search accommodations
place = "Barcelona"
search1 = browser.find_element(by="xpath", value='//*[@id=":rh:"]')
search1.send_keys(place)

In [61]:
# Use ActionChains to press Tab 6 times, to remove the pop-up list that appears
# to select the destination
actions = ActionChains(browser)
for _ in range(6):
    actions.send_keys(Keys.TAB).pause(0.05)
actions.perform()

# Scroll and click on the calendar button
css = "button.ebbedaf8ac:nth-child(2) > span:nth-child(1)"
scroll_and_click(browser = browser, by_type = 'css selector', path = css)

In [62]:
# We select the dates
## Set the path for the date buttons
path = '//div[@id="calendar-searchboxdatepicker"]//table[@class="eb03f3f27f"]//tbody//td[@class="b80d5adb18"]//span[@class="cf06f772fa ef091eb985"]'
## Set the dates (yyyy-mm-dd)
start_date = f"2025-01-24"
end_date = f"2025-01-31"
## Scroll for visibility and click on the desired dates
scroll_and_click_dates(browser, "xpath", path, start_date, end_date)

Now, we should design the code such that:
- It looks for one week shown in the calendar.
- It makes the search.
- It retrieves the desired information from the available accommodations.
- It changes the dates and searches for another week.
- It retrieves the desired information for the new week.
- And successively.

##### Previous date methodology

In [ ]:
# # Path for selecting the dates
# path = '//div[@id="calendar-searchboxdatepicker"]//table[@class="eb03f3f27f"]//tbody//td[@class="b80d5adb18"]//span[@class="cf06f772fa ef091eb985"]'
# # We save all the possible dates that start in the next day when the code is 
# # executed and ends in the last day shown in the 2nd month (on the right)
# dates = browser.find_elements("xpath", path)
# # We check the dates that are being saved
# for date in dates:
#     print(date.get_attribute("data-date"))

# # Input certain dates
# start_day_jan = 24
# end_day_jan = 31

# for date in dates:
#     if date.get_attribute("data-date") == f"2025-01-{start_day_jan}":
#         date.click()
#     if date.get_attribute("data-date") == f"2025-01-{end_day_jan}":
#         date.click()
#         break

# # Looping across the calendar could require additional code.

##### Continuation

In [63]:
# Scroll up until calendar button is visible again and click
css = "button.ebbedaf8ac:nth-child(2) > span:nth-child(1)"
scroll_and_click(browser = browser, by_type = 'css selector', path = css)

In [64]:
# Once we are outside the date selector, we click on the search button
my_xpath = '//div[@id="indexsearch"]//div[@class="ffb9c3d6a3 b3b8f00b52 c9a7790c31 e691439f9a"]//button[@class="a83ed08757 c21c56c305 a4c1805887 f671049264 a2abacf76b c082d89982 cceeb8986b b9fd3c6b3c"]'
browser.find_element(by = "xpath", value = my_xpath).click()

Let's try to get hotel names. To extend this, the idea should be to do a loop where:
1. We scroll down as much as possible, until no new hotels are loaded (i.e., until we reach the button "Cargar más resultados").
2. We click the button "Cargar más resultados".
3. We scroll down again with the same idea, until we can click the reload button.
4. And successively until there are no new accommodations that can be loaded.
5. Once all of the accomodations have been loaded, we extract all the accomodation info.

In [65]:
# We click on the cross to dismiss the Genius pop-up that appears prompting for
# signing up or signing in

# TODO: make this click optional (only if the button exists!)

path = '//div[@class="f0c216ee26 c676dd76fe b5018b639f"]//button[@class="a83ed08757 c21c56c305 f38b6daa18 d691166b09 ab98298258 f4552b6561"]'
browser.find_element(by="xpath", value = path).click()

In [ ]:
path = '//div[@class="c82435a4b8 f581fde0b8"]//button[@class="a83ed08757 c21c56c305 bf0537ecb5 f671049264 af7297d90d c0e0affd09"]'
# If it doesn't load all results, try increasing the scroll pause (in seconds)
# Efficiency doesn't seem to increase significantly by decreasing the number of 
# max_attempts
load_all_results(browser = browser, 
                 button_xpath = path, 
                 scroll_pause=0.25, max_attempts=10)

Reached maximum attempts or button not found.
Total 'Load more results' button clicks: 25.


##### Alternatives
The alternatives below work, but they are more complex and/or less efficient.

Alternative with timeout:

In [ ]:
# Scroll down and click in the Load more results" button

# Initialize variables
no_action_start_time = time.time()
timeout_seconds = 5 # Seconds for exiting the loop since the last action
click_count = 0  # Counter for the number of button clicks

# Define the XPath for the "Load more results" button
path = '//div[@class="c82435a4b8 f581fde0b8"]//button[@class="a83ed08757 c21c56c305 bf0537ecb5 f671049264 af7297d90d c0e0affd09"]'

# Loop until no action for `timeout_seconds` or button is no longer visible
while True:
    if scroll_until_button_visible(browser, path):
        scroll_and_click(browser=browser, by_type='xpath', path=path)
        click_count += 1
        
        # Reset the no-action timer after a successful click
        no_action_start_time = time.time()
    else:
        print("Load more results button not found.")
        break  # Exit the loop if no button is found

    # Check if no action has occurred within the timeout period
    if time.time() - no_action_start_time > timeout_seconds:
        print("No new results loaded for 5 seconds. Exiting loop.")
        break

# Print the final click count
print(f"Total 'Load more results' button clicks: {click_count}.")

In this case, it exits the loop after clicking on the button "Load more results" whenever, after scrolling, the last scroll position is the same as the initial one (so it has reached the end of the page). 

In [230]:
# Initialize variables
no_action_start_time = time.time()
timeout_seconds = 5  # Seconds for exiting the loop since the last action
click_count = 0  # Counter for the number of button clicks
previous_scroll_position = None  # Variable to track the scroll position

# Define the XPath for the "Load more results" button
path = '//div[@class="c82435a4b8 f581fde0b8"]//button[@class="a83ed08757 c21c56c305 bf0537ecb5 f671049264 af7297d90d c0e0affd09"]'

# Loop until no action for `timeout_seconds` or button is no longer visible
while True:
    # Get current scroll position
    current_scroll_position = browser.execute_script("return window.pageYOffset;")
    
    if current_scroll_position == previous_scroll_position:
        print("No new results loaded after scroll. Exiting loop.")
        break  # Exit the loop if scroll position hasn't changed (no new content loaded)
    
    # Update the previous scroll position
    previous_scroll_position = current_scroll_position
    
    if scroll_until_button_visiblev3(browser, path):
        scroll_and_click(browser=browser, by_type='xpath', path=path)
        click_count += 1
        print(f"Clicked the 'Load more results' button. Total clicks so far: {click_count}.")
        
        # Reset the no-action timer after a successful click
        no_action_start_time = time.time()
    else:
        print("Load more results button not found.")
        break  # Exit the loop if no button is found

    # Check if no action has occurred within the timeout period
    if time.time() - no_action_start_time > timeout_seconds:
        print("No new results loaded for 5 seconds. Exiting loop.")
        break

# Print the final click count
print(f"Total 'Load more results' button clicks: {click_count}.")

Load more results button found and visible.
Clicked the 'Load more results' button. Total clicks so far: 1.
Load more results button found and visible.
Clicked the 'Load more results' button. Total clicks so far: 2.
Load more results button found and visible.
Clicked the 'Load more results' button. Total clicks so far: 3.
Load more results button found and visible.
Clicked the 'Load more results' button. Total clicks so far: 4.
Load more results button found and visible.
Clicked the 'Load more results' button. Total clicks so far: 5.
Load more results button found and visible.
Clicked the 'Load more results' button. Total clicks so far: 6.
Load more results button found and visible.
Clicked the 'Load more results' button. Total clicks so far: 7.
Load more results button found and visible.
Clicked the 'Load more results' button. Total clicks so far: 8.
Load more results button found and visible.
Clicked the 'Load more results' button. Total clicks so far: 9.
Load more results button fou

##### Continuation

In [68]:
hotel_elements = browser.find_elements("xpath", '//div[@class="f6431b446c a15b38c233"]')
hotel_elements

[<selenium.webdriver.remote.webelement.WebElement (session="967d739b-025d-437c-9d04-852b0e2321aa", element="84e0d90c-3d62-43c8-b3d9-070b08e6aafd")>,
 <selenium.webdriver.remote.webelement.WebElement (session="967d739b-025d-437c-9d04-852b0e2321aa", element="9737475b-1034-4707-8c9b-208eaefc2988")>,
 <selenium.webdriver.remote.webelement.WebElement (session="967d739b-025d-437c-9d04-852b0e2321aa", element="d7e36f30-2852-4a08-ae4c-23e64f824177")>,
 <selenium.webdriver.remote.webelement.WebElement (session="967d739b-025d-437c-9d04-852b0e2321aa", element="e15836e6-1788-48cb-ae0b-c9d695e5e0d5")>,
 <selenium.webdriver.remote.webelement.WebElement (session="967d739b-025d-437c-9d04-852b0e2321aa", element="7b5bc909-81f2-4b1a-b72c-504b38c9d9db")>,
 <selenium.webdriver.remote.webelement.WebElement (session="967d739b-025d-437c-9d04-852b0e2321aa", element="75b2db85-b0bb-47f9-98fb-1df6008fe4f1")>,
 <selenium.webdriver.remote.webelement.WebElement (session="967d739b-025d-437c-9d04-852b0e2321aa", element

WARNING! The number of accommodations it saves does not completely match the number of results of Booking (but by a small margin, like 664 accommodations out of 676 found). However, sometimes it gets all of the accommodations.

## Can you guess the same for ratings and hotels?

In [85]:
# Method that was in the notebook
# ratings = browser.find_elements('xpath', '/html/body/div[4]/div/div/div/div[2]/div[3]/div[2]/div[2]/div[3]/div[22]/div[1]/div[2]/div/div[1]/div[2]/div/div/a/span/div/div[1]/div')
# hotels = browser.find_elements('xpath','/html/body/div[4]/div/div/div/div[2]/div[3]/div[2]/div[2]/div[3]/div[9]/div[1]/div[2]/div/div[1]/div[1]/div/div[1]/div/h3/a/div[1]')

In [69]:
hotel_names = [element.text for element in hotel_elements]
hotel_names

['Sunotel Central',
 'Sonder Los Arcos',
 'Tembo Barcelona',
 'Vincci Bit',
 'Moxy Barcelona',
 'Motel One Barcelona-Ciutadella',
 'Fisa Rentals Ramblas Apartments',
 'Vincci Maritimo',
 'Catalonia Sagrada Familia',
 'Hotel Viladomat',
 'Hotel Market',
 'Travelodge Barcelona Poblenou',
 'Vincci Mae',
 'Catalonia La Boquería',
 'NH Barcelona Diagonal Center',
 'Leonardo Hotel Barcelona Gran Via',
 'Hotel Astoria',
 'Catalonia Atenas',
 'Hotel Condal',
 'Catalonia Barcelona 505',
 'Chic & Basic Velvet',
 'Pol & Grace Hotel',
 'Hotel America Barcelona',
 'Sallés Hotel Pere IV',
 'Golden Hotel Barcelona',
 'NH Sants Barcelona',
 'Silken Sant Gervasi',
 'Hotel Barcelona Universal',
 'Catalonia Gracia',
 'H Regas Adults Only',
 'Catalonia Diagonal Centro',
 'Bonanova Suite',
 'Lamaro Hotel',
 'Ilunion Les Corts Spa',
 'Sunotel Junior',
 'Acevi Villarroel',
 'ibis Styles Barcelona City Bogatell',
 'AC Hotel Sants by Marriott',
 'Hotel Monegal',
 'Oriente Atiram',
 'Hotel Santa Marta',
 'Leona

In [25]:
rating_elements = browser.find_elements(
    "xpath", '//div[@class="a3b8729ab1 d86cee9b25"]'
)
ratings = [element.text for element in rating_elements]
ratings

['Puntuación: 9,4\n9,4',
 'Puntuación: 8,0\n8,0',
 'Puntuación: 7,4\n7,4',
 'Puntuación: 8,6\n8,6',
 'Puntuación: 8,6\n8,6',
 'Puntuación: 8,0\n8,0',
 'Puntuación: 8,1\n8,1',
 'Puntuación: 8,1\n8,1',
 'Puntuación: 8,3\n8,3',
 'Puntuación: 8,5\n8,5',
 'Puntuación: 8,5\n8,5',
 'Puntuación: 7,8\n7,8',
 'Puntuación: 8,1\n8,1',
 'Puntuación: 8,3\n8,3',
 'Puntuación: 8,3\n8,3',
 'Puntuación: 8,4\n8,4',
 'Puntuación: 8,2\n8,2',
 'Puntuación: 8,2\n8,2',
 'Puntuación: 8,4\n8,4',
 'Puntuación: 8,2\n8,2',
 'Puntuación: 7,7\n7,7',
 'Puntuación: 8,3\n8,3',
 'Puntuación: 8,8\n8,8',
 'Puntuación: 8,2\n8,2',
 'Puntuación: 7,5\n7,5',
 'Puntuación: 7,3\n7,3',
 'Puntuación: 8,4\n8,4',
 'Puntuación: 8,5\n8,5',
 'Puntuación: 8,3\n8,3',
 'Puntuación: 8,1\n8,1',
 'Puntuación: 8,5\n8,5',
 'Puntuación: 8,8\n8,8',
 'Puntuación: 8,4\n8,4',
 'Puntuación: 8,2\n8,2',
 'Puntuación: 7,3\n7,3',
 'Puntuación: 8,4\n8,4',
 'Puntuación: 7,6\n7,6',
 'Puntuación: 8,0\n8,0',
 'Puntuación: 8,4\n8,4',
 'Puntuación: 7,9\n7,9',


In [26]:
# Now we only keep the last 3 characters in each string, thus keeping
# only the rating
ratings_clean = [rating[-3:] for rating in ratings]

# We substitute the commas with dots convert the ratings to floats
ratings_floats = [float(rating.replace(",", ".")) for rating in ratings_clean]
ratings_floats

[9.4,
 8.0,
 7.4,
 8.6,
 8.6,
 8.0,
 8.1,
 8.1,
 8.3,
 8.5,
 8.5,
 7.8,
 8.1,
 8.3,
 8.3,
 8.4,
 8.2,
 8.2,
 8.4,
 8.2,
 7.7,
 8.3,
 8.8,
 8.2,
 7.5,
 7.3,
 8.4,
 8.5,
 8.3,
 8.1,
 8.5,
 8.8,
 8.4,
 8.2,
 7.3,
 8.4,
 7.6,
 8.0,
 8.4,
 7.9,
 8.9,
 6.5,
 7.6,
 9.2,
 8.3,
 8.3,
 7.3,
 8.6,
 8.8,
 8.5,
 8.0,
 8.1,
 8.6,
 8.3,
 8.4,
 7.3,
 8.0,
 8.4,
 8.2,
 8.5,
 7.6,
 8.2,
 8.0,
 8.5,
 8.8,
 8.2,
 8.4,
 8.8,
 8.4,
 8.8,
 8.4,
 8.4,
 8.2,
 8.5,
 8.2,
 8.9,
 8.5,
 8.8,
 8.4,
 7.6,
 8.1,
 8.0,
 8.1,
 8.8,
 8.4,
 7.9,
 8.2,
 8.5,
 8.6,
 7.7,
 8.5,
 7.7,
 8.4,
 8.5,
 8.6,
 8.1,
 8.0,
 8.3,
 8.2,
 7.6]

In [62]:
# Method of the TA for extracting the text from the Selenium objects containing
# the hotel names and ratings
# ratings_list=[]
# for i in ratings:
#     print(i.text)
#     ratings_list.append(i.text)
# hotels_list=[]
# for i in hotels:
#     print(i.text)
#     hotels_list.append(i.text)

8,6
8,7
8,0
8,0
7,9
8,2
7,7
7,1
8,9
8,5
8,2
8,8
8,3
8,4
7,5
8,6
6,5
8,1
8,6
8,7
8,0
8,3
7,1
7,7
8,3
8,7
Hotel Pinar Plaza
Sonder Santa Ana
ibis budget Madrid Aeropuerto
Art Seven Hostel Capsules
Hostal Goyal Pizarro
AmazINN Places Palacio Real
Ibis Budget Madrid Calle 30
AmazINN Places Chamberi
SmartRental Collection Gran Vía Capital
NH Madrid Ribera del Manzanares
Ibis Madrid Calle Alcalá
AmazINN Places Plaza de España 2
Smartrental Madrid Reina Sofia
Axel Hotel Madrid - Adults Only
Ibis Budget Madrid Vallecas
Akeah Hotel Gran Vía
flor hostel capsules
easyHotel Madrid Centro Atocha
Axor Barajas
CriteriaHome! nice&easy
Exclusive Apartment in Historic Center
Apartamentos Matute 11
Apartamentos Day Madrid SILVA Centro Gran Via Sol Malasaña
Hostal Continental
Luz Madrid Rooms
diezmadrid
Estudio entero • 1 baño • 1 cocina • 25m²
Habitación compartida
Apartamento entero • 1 dormitorio • 1 baño • 1 cocina • 35m²
Apartamento entero • 1 dormitorio • 1 sala de estar • 1 baño • 1 cocina • 42m²
A

## You will need the number of pages too!

In [70]:
def get_number_pages(browser):
    """
    Get the number of pages.
    """
    a = browser.find_elements("xpath", "???")
    return int(a[-1].text)


pages = get_number_pages(browser)

print(pages)

40
